In [1]:
"""
Autoregressive (AR) Forecasting Model
=======================================
Real-time recursive forecasts of log real TTF NG prices.
Specifications: AR(1), AR(12), AR(AIC, p≤6)
Iterated multi-step forecasting. Expanding window from Feb 2006.
"""

import numpy as np
import pandas as pd
from statsmodels.tsa.ar_model import AutoReg
import warnings
warnings.filterwarnings("ignore")

In [2]:
# ── Parameters ────────────────────────────────────────────────────────────────
HORIZONS   = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START = "2015-01-01"
AIC_P_MAX  = 6
INPUT_FILE = "Input_TTF_NG_Real_Average_Prices.xlsx"
OUTPUT_FILE = "Output_AR_forecasts.xlsx"

In [3]:
# AR specifications: (label, fixed_p or None for AIC)
SPECIFICATIONS = [
    ("AR(1)",        1),
    ("AR(12)",       12),
    ("AR(AIC,p≤6)", None),   # None triggers AIC selection
]

In [4]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, parse_dates=["date"])
df = df[["date", "price_real"]].sort_values("date").reset_index(drop=True)
df["log_price"] = np.log(df["price_real"])


In [5]:
# ── Helper: actual real price for a given year-month string ──────────────────
def get_actual(ym_str):
    match = df[df["date"].dt.to_period("M").astype(str) == ym_str]
    return match["price_real"].values[0] if len(match) == 1 else np.nan

In [6]:
# ── Helper: select lag order via AIC (recursive, capped at AIC_P_MAX) ────────
def select_aic_lag(y_history, p_max):
    best_aic, best_p = np.inf, 1
    for p in range(1, p_max + 1):
        try:
            m = AutoReg(y_history, lags=p, old_names=False).fit()
            if m.aic < best_aic:
                best_aic, best_p = m.aic, p
        except Exception:
            pass
    return best_p

In [7]:
# ── Helper: iterated h-step forecast from AR(p) ──────────────────────────────
def iterated_forecast(y_history, p, h_max):
    model  = AutoReg(y_history, lags=p, old_names=False).fit()
    params = model.params          # [intercept, phi_1, ..., phi_p]
    intercept = params[0]
    phis      = params[1:]         # length p

    # Buffer: start with last p actual values, then append forecasts
    buffer = list(y_history[-p:])

    forecasts = {}
    for h in range(1, h_max + 1):
        # AR(p) one-step: c + phi_1*y_{t-1} + ... + phi_p*y_{t-p}
        y_hat = intercept + np.dot(phis, buffer[-p:][::-1])
        buffer.append(y_hat)
        if h in HORIZONS:
            forecasts[h] = y_hat

    return forecasts

In [8]:
# ── Main forecasting loop ─────────────────────────────────────────────────────
records = []
origins = df[df["date"] >= EVAL_START]["date"].tolist()
h_max   = max(HORIZONS)

for origin_date in origins:

    # All log prices up to and including the forecast origin
    history = df[df["date"] <= origin_date]["log_price"].values
    origin_log_price = history[-1]   # log price at forecast origin

    for label, fixed_p in SPECIFICATIONS:

        # Determine lag order
        if fixed_p is not None:
            p = fixed_p
        else:
            p = select_aic_lag(history, AIC_P_MAX)

        # Need at least p+1 observations to fit AR(p)
        if len(history) <= p:
            continue

        # Iterated forecasts for all horizons
        try:
            forecasts = iterated_forecast(history, p, h_max)
        except Exception:
            continue

        # Record results
        for h in HORIZONS:
            if h not in forecasts:
                continue

            actual_ym      = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")
            forecast_level = np.exp(forecasts[h])   # convert log → level
            actual_val     = get_actual(actual_ym)

            records.append({
                "forecast_origin": origin_date.strftime("%Y-%m-%d"),
                "horizon":         h,
                "model":           label,
                "actual_month":    actual_ym,
                "forecast":        forecast_level,
                "actual":          actual_val,
                "lag_order_used":  p,           # useful for AR(AIC) audit
            })

In [9]:
# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

In [10]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("AR Forecasting complete.")
print(f"  Specifications:   {[s[0] for s in SPECIFICATIONS]}")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Horizons:         {HORIZONS}")
print(f"  Total rows:       {len(results)}")
print(f"  Output saved to:  {OUTPUT_FILE}")
print()

AR Forecasting complete.
  Specifications:   ['AR(1)', 'AR(12)', 'AR(AIC,p≤6)']
  Forecast origins: 132
  Horizons:         [1, 3, 6, 9, 12, 15, 18, 21, 24]
  Total rows:       3564
  Output saved to:  Output_AR_forecasts.xlsx



In [11]:
# Sample output — first origin, all models
first_origin = results["forecast_origin"].min()
sample = results[results["forecast_origin"] == first_origin]
print(f"Sample — first origin ({first_origin}):")
print(sample[["model","horizon","lag_order_used","actual_month",
              "forecast","actual"]].to_string(index=False))

Sample — first origin (2015-01-31):
      model  horizon  lag_order_used actual_month  forecast    actual
      AR(1)        1               1      2015-02 19.808950 22.938516
      AR(1)        3               1      2015-04 19.877842 22.046423
      AR(1)        6               1      2015-07 19.960848 20.679393
      AR(1)        9               1      2015-10 20.024476 18.160884
      AR(1)       12               1      2016-01 20.073209 13.882111
      AR(1)       15               1      2016-04 20.110508 12.101522
      AR(1)       18               1      2016-07 20.139042 14.115551
      AR(1)       21               1      2016-10 20.160862 15.958961
      AR(1)       24               1      2017-01 20.177543 19.873384
     AR(12)        1              12      2015-02 19.641507 22.938516
     AR(12)        3              12      2015-04 19.513725 22.046423
     AR(12)        6              12      2015-07 19.290846 20.679393
     AR(12)        9              12      2015-10 20.7

In [12]:
# AR(AIC) lag order distribution — useful for dissertation
aic_df = results[results["model"] == "AR(AIC,p≤6)"].copy()
if not aic_df.empty:
    aic_summary = (aic_df.drop_duplicates("forecast_origin")
                         ["lag_order_used"].value_counts().sort_index())
    print("\nAR(AIC) lag order distribution across forecast origins:")
    print(aic_summary.to_string())
    print(f"  Most common: AR({aic_summary.idxmax()})")


AR(AIC) lag order distribution across forecast origins:
lag_order_used
1    15
2    25
3    16
4     3
5    73
  Most common: AR(5)
